## Experiment Workflow Test
Short notebook to validate the new experiment file workflow (`experiments/podmr.py` and `experiments/rabi.py`).

In [ ]:
# Adjust if your VS Code kernel needs help finding pip / scripts
import os
python_path = r"C:\Users\QT3 User Facility\AppData\Local\Programs\Python\Python314\Scripts"
os.environ["PATH"] = python_path + ";" + os.environ["PATH"]

!pip --version
!pip show qickdawg

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import matplotlib.pyplot as plt
from copy import copy
from scipy.optimize import curve_fit
import qickdawg as qd
from importlib.metadata import version

print(f"numpy     {version('numpy')}")
print(f"qickdawg  {version('qickdawg')}")
print(f"python    {sys.version}")

### Connect to board

In [ ]:
qd.start_client('192.168.3.1')
print("Connected — soccfg loaded:", qd.soccfg is not None)

### Laser on / off check

In [ ]:
default_config = qd.NVConfiguration()

default_config.adc_channel              = 0
default_config.mw_channel               = 0
default_config.mw_nqz                   = 1
default_config.mw_gain                  = 1800
default_config.laser_gate_pmod          = 0
default_config.laser_on_tus             = 2
default_config.laser_readout_offset_tus = 1.172
default_config.readout_integration_tns  = 300
default_config.mw_laser_delay_treg      = 0
default_config.relax_delay_tus          = 1.1
default_config.pre_init                 = True

# Expected: laser should illuminate the sample
qd.laser_on(default_config)

In [ ]:
# Expected: laser should turn off
qd.laser_off(default_config)

### PL counts — live scope

In [ ]:
config = copy(default_config)
config.readout_integration_treg = 2**16 - 1
config.reps = 1

prog = qd.PLIntensity(config)

def get_cps():
    d = prog.acquire(progress=False)
    return d / qd.max_int_time_treg / qd.min_time_tns * 1e9

qd.live_plot(get_cps)

### Run PODMR experiment script
Edit parameters in `experiments/podmr.py` (EXPERIMENT PARAMETERS block), then run this cell.
The script handles connection, sweep config, acquisition, HDF5 save, and plot internally.
Use `%run -i` to inject variable overrides from a prior cell without editing the file.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../experiments"))

%run ../experiments/podmr.py

#### Quick inline PODMR re-plot
`%run` leaves `data`, `config`, `x_mhz`, and `timestamp` in the kernel namespace.
Re-plot here without re-acquiring to iterate on visuals.

In [ ]:
# ratio plot — works whether data is (nsweep,) or (2, nsweep)
if data.ndim == 2 and data.shape[0] == 2:
    ratio = data[0] / data[1]
else:
    ratio = data

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_mhz, ratio, marker="o", markersize=3, linewidth=1.2)
ax.set_xlabel("MW Frequency (MHz)")
ax.set_ylabel("Contrast (ratio)")
ax.set_title(f"pODMR quick view — {timestamp}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Run Rabi experiment script
Edit parameters in `experiments/rabi.py`, then run this cell.

In [ ]:
%run ../experiments/rabi.py

#### Quick inline Rabi re-plot
`%run` leaves `data`, `config`, `x_ns`, and `timestamp` in the kernel namespace.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_ns, data, marker="o", markersize=3, linewidth=1.2)
ax.set_xlabel("MW Pulse Duration (ns)")
ax.set_ylabel("Signal (ADC counts)")
ax.set_title(f"Rabi quick view — {timestamp}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Save PODMR result to CSV (workflow verification)

In [ ]:
import pandas as pd
from pathlib import Path

csv_dir = Path("../data/podmr")
csv_dir.mkdir(parents=True, exist_ok=True)

if data.ndim == 2 and data.shape[0] == 2:
    sig, ref = data[0], data[1]
else:
    sig, ref = data, np.ones_like(data)

df = pd.DataFrame({
    "frequency_MHz": x_mhz,
    "signal":        sig,
    "reference":     ref,
    "ratio":         sig / ref,
})

csv_path = csv_dir / f"podmr_workflow_test_{timestamp}.csv"
df.to_csv(csv_path, index=False)
print(f"Saved → {csv_path.resolve()}")